In [24]:
import os
from nwtrace import *
import pandas as pd
import geopandas as gpd

network_path = "data/more/full_sewers.geojson"
node_path = "data/more/full_nodes.geojson"
project_crs = "EPSG:26717"

multiple = True
upstream_only = True
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROM'
downstream_field = 'TO'

search_distance = 0.1

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

In [25]:

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls
# target_endpoints = ["OF3729806115"]

outputname_extra = "BC_all_outlets_"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

In [3]:
sewers = gpd.read_file(network_path)
nodes = gpd.read_file(node_path).rename(columns={"FACILITYID": 'node_id'})

In [4]:
# id_field = 'node_id'
# upstream_only = False
# downstream_only = True

# inlets = nodes[nodes['layer'] == "ssINLET"]
# inlets = inlets[id_field].to_list()

# target_endpoints = inlets


In [5]:
repaired_network = repair.repair_segment_connections(
    segments=sewers,
    nodes=nodes,
    segment_id_field=sewer_id_field,
    node_id_field='node_id',
    upstream_field=upstream_field,
    downstream_field=downstream_field,
    distance_threshold=search_distance
)

In [6]:
repaired_network = repaired_network.drop_duplicates(subset=sewer_id_field)

In [7]:
repaired_nodes = repair.repair_connections(
    nodes,
    repaired_network,
    'node_id',
    sewer_id_field,
    "TO_FIXED",
    reference_connection_fields=[upstream_field, downstream_field],
    distance_threshold=search_distance
)

In [8]:

sewershed = NWTrace(
    network=repaired_network,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
    crs=project_crs
)

In [9]:
fittings = gpd.read_file("data/more/fitting_connections.geojson")
# catchbasin_leads = gpd.read_file("data/more/catchbasin_leads.gpkg")

In [10]:

# # additional connections
nodes_up = (repaired_nodes[["node_id", "TO_FIXED", "geometry"]]
            .rename(columns={"node_id": 'node_id', "TO_FIXED": 'segment_id'})
            .set_index('node_id', drop=False).to_dict(orient="index"))

sewershed.add_upstream_nodes(nodes_up, search_geometry=False)


# new_segs = (catchbasin_leads[["FACILITYID", "UP_ASSET_ID", "DN_ASSET_ID"]]
#             .rename(columns={"FACILITYID": 'segment_id', "UP_ASSET_ID": 'from', "DN_ASSET_ID": 'to'})
#             .set_index('segment_id').to_dict(orient="index"))

# sewershed.add_segments(new_segs)

Added 284535 node-segment connection(s)
Created 2491 new node(s)
Created 64 new segment(s).



In [11]:
BC_nodes = gpd.read_file('data/more/BC_nodes.geojson')

In [26]:
from_nodes = set(repaired_network[upstream_field].dropna())
to_nodes = set(repaired_network[downstream_field].dropna())

valid_nodes = set(BC_nodes['FACILITYID'])

endpoints = (to_nodes - from_nodes) & valid_nodes

endpoints_gdf = BC_nodes[BC_nodes['FACILITYID'].isin(endpoints)]

id_field = "FACILITYID"
target_endpoints = endpoints_gdf[id_field].to_list()

In [27]:
if multiple == False:
    result = sewershed.trace_sewershed(
        target_endpoints[0], 
        upstream_only=upstream_only, 
        downstream_only=downstream_only
    )
else:
    result = sewershed.trace_sewersheds(
        target_endpoints, 
        upstream_only=upstream_only, 
        downstream_only=downstream_only, 
    )

Tracing Sewer Network from endpoint(s) [CN4899(...)]
	Direction(s): upstream
Preparing directional node connection tree...
Searching Network:


100%|██████████| 473/473 [00:00<00:00, 3153.55it/s]


Found 14 connections overall to all 473 endpoints
Finished!


In [28]:
d_node, d_seg = sewershed.get_directional_lookup_tables()

In [29]:
d_seg['CL9478']

{'from': ['CB3838007097', 'CB3838007097'], 'to': ['MH3836707104']}

In [30]:
d_node['MH3873708965']

{'in': ['SL51551', 'SL51542', 'SL53208'], 'out': ['SL53230', 'SL53230']}

In [31]:
sewershed_network = pd.DataFrame.from_dict(result).rename(columns={'segment_id': sewer_id_field})

# TODO: Should make consistent with rest of code and reset index by default in any function that sets it (every function should be index-agnostic)
repaired_network = repaired_network.reset_index(drop=True)
delin_sewershed_network = repaired_network.merge(sewershed_network, on=sewer_id_field, how="right")

delin_sewershed_network.to_file(
    f'{output_dir}/{outputname_extra}catchment_'
    f'{"singledir" if upstream_only or downstream_only else "multidir"}'
    f'{"_ups" if upstream_only and not downstream_only else ""}'
    f'{"_dwns" if downstream_only and not upstream_only else ""}_'
    f'{target_endpoints[0] if not multiple else outfall_file.replace("/", "_").replace(".", "_")}.geojson'
)